In [2]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf

In this warmup, we're going to build a model looking at the probability that it rains in a given day.

In [3]:
rain = pd.read_csv("../data/rainfall.csv")
rain.head(2)

,date,rainfall_in
0,2008-01-01,0.0
1,2008-01-02,0.0


First, create an indicator variable for whether it rained on a given day. This variable should be 0 if there was 0 inches of rain and 1 if there was a non-zero amount of rain.

In [6]:
rain['rained'] = (rain['rainfall_in'] > 0).astype(int)

In [9]:
rain.tail(5)

,date,rainfall_in,rained
6165,2024-11-17,0.00,0
6166,2024-11-18,0.00,0
6167,2024-11-19,0.05,1
6168,2024-11-20,0.60,1
6169,2024-11-21,0.00,0


Now, use the [shift method](https://pandas.pydata.org/docs/reference/api/pandas.Series.shift.html) to create a "rain_yesterday" column giving the rainfall total for the previous day.

In [41]:
rain['rain_yesterday'] = rain["rained"].shift(periods=1)
rain.head()

,date,rainfall_in,rained,rain_yesterday,day_name
0,2008-01-01,0.0,0,NaN,Tuesday
1,2008-01-02,0.0,0,0.0,Wednesday
2,2008-01-03,0.0,0,0.0,Thursday
3,2008-01-04,0.0,0,0.0,Friday
4,2008-01-05,0.0,0,0.0,Saturday


Create a logistic regression model estimating the probability that it rains today based on the amount of rain yesterday.

Is the coefficient for rain_yesterday statistically significant? Hint: look at the summary of the fit model.

In [12]:
rain_model = smf.logit(formula="rained ~ rain_yesterday", data=rain).fit()

Optimization terminated successfully.
         Current function value: 0.645186
         Iterations 5


In [14]:
rain_model.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                           Logit Regression Results                           
==============================================================================
Dep. Variable:                 rained   No. Observations:                 6169
Model:                          Logit   Df Residuals:                     6167
Method:                           MLE   Df Model:                            1
Date:                Sat, 13 Dec 2025   Pseudo R-squ.:                 0.04138
Time:                        09:21:34   Log-Likelihood:                -3980.2
converged:                       True   LL-Null:                       -4152.0
Covariance Type:            nonrobust   LLR p-value:                 1.030e-76
==================================================================================
                     coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------------------------
Intercept         -0.8208      0.036    -23.003      0.000      -0.891      -0.751
rain_yesterday     0.9881      0.054     18.332      0.000       0.882       1.094
==================================================================================
"""

### ANSWER: Based on the p-value, the predictor is significant.

Convert the date column to a datetime type and use that to create a column showing the day of the week. 

In [16]:
rain['date'] = pd.to_datetime(rain['date'], format="%Y-%m-%d")

In [18]:
rain['day_name'] = rain['date'].dt.day_name()

Now fit a logistic regression model estimating the probability of it raining on a given day based on the day of the week.

In [19]:
rain_day_model = smf.logit(formula="rained ~ day_name", data=rain).fit()

Optimization terminated successfully.
         Current function value: 0.672172
         Iterations 4


In [20]:
rain_day_model.params

Intercept               -0.450128
day_name[T.Monday]       0.229027
day_name[T.Saturday]     0.085177
day_name[T.Sunday]      -0.038360
day_name[T.Thursday]     0.017186
day_name[T.Tuesday]      0.021935
day_name[T.Wednesday]   -0.006630
dtype: float64

In [21]:
rain_day_model.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                           Logit Regression Results                           
==============================================================================
Dep. Variable:                 rained   No. Observations:                 6170
Model:                          Logit   Df Residuals:                     6163
Method:                           MLE   Df Model:                            6
Date:                Sat, 13 Dec 2025   Pseudo R-squ.:                0.001248
Time:                        09:29:48   Log-Likelihood:                -4147.3
converged:                       True   LL-Null:                       -4152.5
Covariance Type:            nonrobust   LLR p-value:                    0.1101
=========================================================================================
                            coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------------------------------------------
Intercept                -0.4501      0.069     -6.515      0.000      -0.586      -0.315
day_name[T.Monday]        0.2290      0.097      2.366      0.018       0.039       0.419
day_name[T.Saturday]      0.0852      0.097      0.875      0.381      -0.106       0.276
day_name[T.Sunday]       -0.0384      0.098     -0.392      0.695      -0.230       0.154
day_name[T.Thursday]      0.0172      0.098      0.176      0.860      -0.174       0.208
day_name[T.Tuesday]       0.0219      0.098      0.225      0.822      -0.169       0.213
day_name[T.Wednesday]    -0.0066      0.098     -0.068      0.946      -0.198       0.185
=========================================================================================
"""

Is the coefficient for Tuesday statistically significant?

### ANSWER: Tuesaday is not statistically significant

Is the coefficient for Monday statistically significant? What might be wrong with interpreting this p-value directly? (Hint: see [Bonferroni correction](https://en.wikipedia.org/wiki/Bonferroni_correction))

In [42]:
print(f"New p-val: {0.05/6:.6f}")

New p-val: 0.008333


### ANSWER: After using the Bonferroni correction, Monday is not statistically significant